# Routers
> Route queries to the right retriever based on rules, keywords, or metadata.

| Module | `anchor.retrieval` |
|--------|--------------------|
| Key classes | `CallbackRouter`, `KeywordRouter`, `MetadataRouter`, `RoutedRetriever` |
| Difficulty | Intermediate |

Routers decide which retriever should handle a query. This enables specialized
retrieval paths for different query types.

**Time:** ~8 minutes

## Setup

In [ ]:
from anchor.retrieval import (
    CallbackRouter,
    KeywordRouter,
    MetadataRouter,
    RoutedRetriever,
    DenseRetriever,
    SparseRetriever,
)
from anchor.storage import InMemoryVectorStore, InMemoryContextStore
from anchor.models import ContextItem, SourceType, QueryBundle

## Create Two Specialized Retrievers
One for code topics, one for general docs.

In [ ]:
def embed_fn(text: str) -> list[float]:
    padded = text[:128].ljust(128)
    return [hash(c) % 100 / 100.0 for c in padded]

# Code retriever
code_store = InMemoryVectorStore()
code_ctx = InMemoryContextStore()
code_retriever = DenseRetriever(
    vector_store=code_store, context_store=code_ctx, embed_fn=embed_fn,
)
code_items = [
    ContextItem(id="code-1", content="def hello(): print('hello world')",
               source=SourceType.RETRIEVAL, score=0.0, priority=5, token_count=8),
    ContextItem(id="code-2", content="async def fetch(url): return await get(url)",
               source=SourceType.RETRIEVAL, score=0.0, priority=5, token_count=10),
]
code_retriever.index(code_items)

# Docs retriever
docs_store = InMemoryVectorStore()
docs_ctx = InMemoryContextStore()
docs_retriever = DenseRetriever(
    vector_store=docs_store, context_store=docs_ctx, embed_fn=embed_fn,
)
docs_items = [
    ContextItem(id="doc-1", content="Anchor is a context management framework.",
               source=SourceType.RETRIEVAL, score=0.0, priority=5, token_count=8),
    ContextItem(id="doc-2", content="Installation: pip install astro-anchor",
               source=SourceType.RETRIEVAL, score=0.0, priority=5, token_count=7),
]
docs_retriever.index(docs_items)

print("Code retriever: 2 items indexed.")
print("Docs retriever: 2 items indexed.")

## CallbackRouter
Route based on a user-defined callback function.

In [ ]:
def route_fn(query: QueryBundle) -> str:
    """Route code-related queries to the code retriever."""
    code_terms = {"def", "async", "function", "class", "code", "implement"}
    query_terms = set(query.query_str.lower().split())
    if query_terms & code_terms:
        return "code"
    return "docs"

callback_router = CallbackRouter(route_fn=route_fn)

# Test routing decisions
for q in ["implement a function", "how to install anchor", "async code example"]:
    route = callback_router.route(QueryBundle(query_str=q))
    print(f"  '{q}' -> route: {route}")

## KeywordRouter
Route based on keyword matching with configurable keyword-to-route mappings.

In [ ]:
keyword_router = KeywordRouter(
    routes={
        "code": ["def", "class", "function", "implement", "code"],
        "docs": ["install", "guide", "tutorial", "documentation"],
    },
    default_route="docs",
)

for q in ["implement a class", "installation guide", "random query"]:
    route = keyword_router.route(QueryBundle(query_str=q))
    print(f"  '{q}' -> route: {route}")

## MetadataRouter
Route based on metadata attached to the `QueryBundle`.

In [ ]:
metadata_router = MetadataRouter(
    metadata_key="domain",
    default_route="docs",
)

q_code = QueryBundle(query_str="show me examples", metadata={"domain": "code"})
q_docs = QueryBundle(query_str="show me examples", metadata={"domain": "docs"})
q_none = QueryBundle(query_str="show me examples")

print(f"  domain='code'  -> route: {metadata_router.route(q_code)}")
print(f"  domain='docs'  -> route: {metadata_router.route(q_docs)}")
print(f"  domain=None    -> route: {metadata_router.route(q_none)} (default)")

## RoutedRetriever
Combine a router with a mapping of route names to retrievers.

In [ ]:
routed = RoutedRetriever(
    router=keyword_router,
    retrievers={
        "code": code_retriever,
        "docs": docs_retriever,
    },
)

# Query routed to 'code' retriever
results = routed.retrieve(QueryBundle(query_str="implement a function"), top_k=2)
print("Query: 'implement a function' (routed to: code)")
for r in results:
    print(f"  {r.id}: {r.content[:50]}")

print()

# Query routed to 'docs' retriever
results = routed.retrieve(QueryBundle(query_str="installation guide"), top_k=2)
print("Query: 'installation guide' (routed to: docs)")
for r in results:
    print(f"  {r.id}: {r.content[:50]}")

## Key Takeaways
- `CallbackRouter` routes via a user-defined function for maximum flexibility
- `KeywordRouter` routes by matching query terms against keyword lists
- `MetadataRouter` routes based on `QueryBundle.metadata` fields
- `RoutedRetriever` wires a router to a dict of retrievers
- Routing enables specialized retrieval paths without complex conditionals

**Next:** [Custom Retriever](09_custom_retriever.ipynb)